In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# 1. Setup paths
data_path = './NIDS/cicids2017/' # Folder containing the 8 CSVs
output_base = './data_split/'

# Create directories
for folder in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_base, folder), exist_ok=True)

# 2. Load and Merge
files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
df_list = []

for file in files:
    temp_df = pd.read_csv(os.path.join(data_path, file), skipinitialspace=True)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

# 3. Preprocessing & Leakage Prevention
# Drop columns that cause "identity" leakage
leakage_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Timestamp']
df.drop(columns=[col for col in leakage_cols if col in df.columns], inplace=True)

# Fix column names (remove spaces/special chars)
df.columns = [c.strip() for c in df.columns]

# Handle Infinite and NaN values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# 4. The Split (60% Train, 20% Val, 20% Test)
X = df.drop('Label', axis=1)
y = df['Label']

# First split: Train and "Temp" (which becomes Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# Second split: Split Temp into Val and Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# 5. Save to Folders
def save_to_csv(X_part, y_part, folder_name):
    combined = pd.concat([X_part, y_part], axis=1)
    combined.to_csv(os.path.join(output_base, folder_name, 'data.csv'), index=False)
    print(f"Saved {folder_name} set with shape: {combined.shape}")

save_to_csv(X_train, y_train, 'train')
save_to_csv(X_val, y_val, 'val')
save_to_csv(X_test, y_test, 'test')

Saved train set with shape: (1696725, 79)
Saved val set with shape: (565575, 79)
Saved test set with shape: (565576, 79)


In [3]:
import os
import numpy as np
import pandas as pd
import shutil
#import cv2
import random
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")

# === CONFIG ===
IMG_SIZE = 224
FEATURE_SIZE = 243  # For 9x9x3 image
IMG_SHAPE = (9, 9, 3)
RESIZED_TRAIN_DIR = './NIDS/Images/train_resized'

In [8]:
TRAIN_DIR = './NIDS/Images/val'


# === Load & Clean Dataset ===
df = pd.read_csv('./data_split/val/data.csv', low_memory=False)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
# === Label Mapping ===
def map_label(label):
    if label == "BENIGN": return "BENIGN"
    elif "DoS" in label: return "DoS"
    elif "PortScan" in label or "Brute Force" in label: return "Fuzzy"
    elif "Web Attack" in label or "Sql Injection" in label or "XSS" in label: return "RPM"
    elif "Bot" in label or "Infiltration" in label or "Heartbleed" in label: return "Advanced"
    elif "Patator" in label: return "Patator"
    else: return "Other"

# Apply mapping
df['Category'] = df['Label'].apply(map_label)
df = df[df['Category'] != "Other"]

# === Feature Scaling ===
features = df.select_dtypes(include=[np.number])
labels = df['Category']
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)
features_scaled = (features_scaled * 255).astype(np.uint8)

# === Create Padded Images ===
def create_image_from_row(row, size=FEATURE_SIZE):
    data = np.array(row)
    if len(data) < size:
        data = np.pad(data, (0, size - len(data)), 'constant')
    else:
        data = data[:size]
    return data.reshape(IMG_SHAPE).astype(np.uint8)

# === Save Images by Class ===
def save_images_by_class(X, y, base_path):
    for idx, (row, label) in enumerate(zip(X, y)):
        img = create_image_from_row(row)
        class_dir = os.path.join(base_path, label)
        os.makedirs(class_dir, exist_ok=True)
        Image.fromarray(img).save(os.path.join(class_dir, f"{idx}.png"))


save_images_by_class(features_scaled, labels, TRAIN_DIR)

print("Images saved by class in train/test folders.")


Images saved by class in train/test folders.


In [6]:
def resize_images(src_dir, dst_dir):
    for root, dirs, files in os.walk(src_dir):
        for name in files:
            src_path = os.path.join(root, name)
            img = cv2.imread(src_path)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            dst_path = root.replace(src_dir, dst_dir)
            os.makedirs(dst_path, exist_ok=True)
            cv2.imwrite(os.path.join(dst_path, name), img)
    print(f"Resized images saved to: {dst_dir}")

resize_images(TRAIN_DIR, RESIZED_TRAIN_DIR)

print("All done.")

NameError: name 'cv2' is not defined

In [9]:
!pip install cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2
